# Multi-Object Tracking Under Occlusion: Results & Performance Analysis

This notebook demonstrates how to load, visualize, and analyze tracking results, MOT benchmark metrics (MOTA, IDF1, ID switches), and failure mode distributions generated by the YOLO + SORT tracking pipeline.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

## 1. Load and Inspect MOT Benchmark Metrics

Load benchmark evaluation metrics (MOTA, IDF1, ID switches, False Positives, False Negatives) saved in `outputs/metrics/metrics.json`.

In [ ]:
metrics_file = "outputs/metrics/metrics.json"

if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        metrics_data = json.load(f)
    print("Loaded metrics from:", metrics_file)
else:
    print("Metrics file not found at", metrics_file)
    print("Displaying representative metric structure:")
    metrics_data = {
        "sequence": "demo_sequence",
        "iou_threshold": 0.5,
        "mota": 0.685,
        "idf1": 0.724,
        "id_switches": 4,
        "false_positives": 12,
        "false_negatives": 48,
        "mostly_tracked": 8,
        "partially_tracked": 2,
        "mostly_lost": 1
    }

df_metrics = pd.DataFrame([metrics_data])
df_metrics.T.rename(columns={0: "Value"})

In [ ]:
# Visualizing Core Performance Metrics
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Accuracy & Identity Consistency
bars1 = ax[0].bar(["MOTA", "IDF1"], [metrics_data.get("mota", 0), metrics_data.get("idf1", 0)], color=["#2563eb", "#10b981"], width=0.5)
ax[0].set_ylim(0, 1.05)
ax[0].set_ylabel("Score [0.0 - 1.0]")
ax[0].set_title("Tracking Accuracy & Identity Preservation")
for b in bars1:
    yval = b.get_height()
    ax[0].text(b.get_x() + b.get_width()/2, yval + 0.02, f"{yval:.3f}", ha="center", va="bottom", weight="bold")

# Tracking Error Counts
err_labels = ["False Positives", "False Negatives", "ID Switches"]
err_vals = [
    metrics_data.get("false_positives", 0),
    metrics_data.get("false_negatives", 0),
    metrics_data.get("id_switches", 0)
]
bars2 = ax[1].bar(err_labels, err_vals, color=["#f59e0b", "#ef4444", "#8b5cf6"], width=0.5)
ax[1].set_ylabel("Count")
ax[1].set_title("Tracking Error Counts")
for b in bars2:
    yval = b.get_height()
    ax[1].text(b.get_x() + b.get_width()/2, yval + 0.5, f"{int(yval)}", ha="center", va="bottom", weight="bold")

plt.tight_layout()
plt.show()

## 2. Failure Analysis & Identity Switches

Load failure cases generated by `scripts/analyze_failures.py` to examine the distribution of causes behind identity switches.

In [ ]:
failure_csv = "outputs/failure_cases/id_switches.csv"

if os.path.exists(failure_csv):
    df_failures = pd.read_csv(failure_csv)
    print(f"Loaded {len(df_failures)} switch events from: {failure_csv}")
else:
    print("Failure CSV not found; showing illustrative breakdown:")
    df_failures = pd.DataFrame([
        {"frame": 45, "gt_id": 2, "prev_tracker_id": 2, "new_tracker_id": 7, "probable_cause": "occlusion", "evidence": "High spatial overlap with adjacent target"},
        {"frame": 88, "gt_id": 5, "prev_tracker_id": 5, "new_tracker_id": 9, "probable_cause": "occlusion", "evidence": "Target missing for 3 frames prior to switch"},
        {"frame": 120, "gt_id": 1, "prev_tracker_id": 1, "new_tracker_id": 8, "probable_cause": "ambiguous association", "evidence": "2 detection candidates with IoU > 0.20"},
        {"frame": 154, "gt_id": 4, "prev_tracker_id": 4, "new_tracker_id": 11, "probable_cause": "fast motion", "evidence": "High centroid displacement: 52.4 px/frame"}
    ])

df_failures

In [ ]:
if not df_failures.empty and "probable_cause" in df_failures.columns:
    cause_counts = df_failures["probable_cause"].value_counts()
    plt.figure(figsize=(7, 4))
    cause_counts.plot(kind="bar", color="#3b82f6", edgecolor="#1e3a8a", width=0.5)
    plt.title("Attributed Causes of Identity Switches")
    plt.xlabel("Probable Cause")
    plt.ylabel("Number of Switches")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## 3. Trajectory Reconstruction & Spatial Paths

Load tracking results (`frame, id, x, y, w, h, conf, ...`) from `outputs/tracking_results/tracking_output.txt` and plot 2D motion paths.

In [ ]:
tracking_file = "outputs/tracking_results/tracking_output.txt"

if os.path.exists(tracking_file):
    df_tracks = pd.read_csv(tracking_file, header=None, names=["frame", "id", "x", "y", "w", "h", "conf", "x3", "y3", "z3"])
    print(f"Loaded {len(df_tracks)} tracking records for {df_tracks['id'].nunique()} unique IDs.")
    
    plt.figure(figsize=(10, 6))
    for track_id, group in df_tracks.groupby("id"):
        if len(group) > 5:  # Filter noise
            cx = group["x"] + group["w"] / 2.0
            cy = group["y"] + group["h"] / 2.0
            plt.plot(cx, cy, label=f"ID {track_id}", linewidth=1.8, marker="o", markersize=2)
    
    plt.title("Reconstructed 2D Target Trajectories")
    plt.xlabel("X Coordinate (pixels)")
    plt.ylabel("Y Coordinate (pixels)")
    plt.gca().invert_yaxis()  # Image coordinate space (top-left is 0,0)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print("No tracking output file found at", tracking_file)
    print("Run `python scripts/run_tracking.py --input <path_to_video>` to produce tracking results.")